In [ ]:
import os
import numpy as np
import pandas as pd

import torch

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from src.audio import extract_logmel
from src.models import MultiModalVAE, MultiModalAE, multimodal_vae_loss, multimodal_ae_loss
from src.evaluation import evaluation_hard
from src.visualisation import plot_training_loss, plot_tsne

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('results_hard/plots', exist_ok=True)

In [ ]:
TSV_PATH = 'data/sampled.tsv'
AUDIO_DIR = 'data/songs'

audio_X, lyrics, genres, languages = extract_logmel(TSV_PATH, AUDIO_DIR)

In [ ]:
# Vectorise lyrics
vectorizer = TfidfVectorizer(max_features=1000)
lyrics_X = vectorizer.fit_transform(lyrics).toarray()

# Normalise
scaler_text = StandardScaler()
lyrics_X = scaler_text.fit_transform(lyrics_X)

In [ ]:
# Normalise spectrogram
audio_X = (audio_X - np.mean(audio_X, axis=(1,2), keepdims=True)) / \
          (np.std(audio_X, axis=(1,2), keepdims=True) + 1e-8)

# Add channel dimension for CNN
audio_X = np.expand_dims(audio_X, axis=1)

# Convert to tensors
audio_tensor = torch.tensor(audio_X, dtype=torch.float32)
lyrics_tensor = torch.tensor(lyrics_X, dtype=torch.float32)

In [ ]:
LATENT_DIM = 32
EPOCHS = 50
BETA = 2

# Text input dimension for model
text_dim = lyrics_X.shape[1]

# Construct Beta-VAE model with beta=2 and use Adam optimiser with learning rate = 1e-3
model_bvae2 = MultiModalVAE(text_dim, latent_dim=LATENT_DIM).to(device)
optimiser = torch.optim.Adam(model_bvae2.parameters(), lr=1e-3)

dataset = torch.utils.data.TensorDataset(audio_tensor, lyrics_tensor)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

loss_history = []

for epoch in range(EPOCHS):
    total_loss = 0

    for (audio, text) in loader:
        audio = audio.to(device)
        text = text.to(device)

        optimiser.zero_grad()

        audio_recon, text_recon, mu, logvar = model_bvae2(audio, text)

        loss = multimodal_vae_loss(audio_recon, audio, text_recon, text, mu, logvar, beta=BETA)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)

    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')

# Save model and loss history
torch.save(model_bvae2.state_dict(), 'results_hard/bvae2.pth')
np.save('results_hard/loss_history_bvae2.npy', loss_history)

In [ ]:
# Plot training loss and save figure
plot_training_loss(loss_history, 'results_hard/plots/loss_curve_bvae2.svg')

In [ ]:
BETA = 4

# Text input dimension for model
text_dim = lyrics_X.shape[1]

# Construct Beta-VAE model with beta=4 and use Adam optimiser with learning rate = 1e-3
model_bvae4 = MultiModalVAE(text_dim, latent_dim=LATENT_DIM).to(device)
optimiser = torch.optim.Adam(model_bvae4.parameters(), lr=1e-3)

dataset = torch.utils.data.TensorDataset(audio_tensor, lyrics_tensor)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

loss_history = []

for epoch in range(EPOCHS):
    total_loss = 0

    for (audio, text) in loader:
        audio = audio.to(device)
        text = text.to(device)

        optimiser.zero_grad()

        audio_recon, text_recon, mu, logvar = model_bvae4(audio, text)

        loss = multimodal_vae_loss(audio_recon, audio, text_recon, text, mu, logvar, beta=BETA)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history.append(avg_loss)

    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')

# Save model and loss history
torch.save(model_bvae4.state_dict(), 'results_hard/bvae4.pth')
np.save('results_hard/loss_history_bvae4.npy', loss_history)

In [ ]:
# Plot training loss and save figure
plot_training_loss(loss_history, 'results_hard/plots/loss_curve_bvae4.svg')

In [ ]:
EPOCHS=50

# Construct AE model and use Adam optimiser with learning rate = 1e-3
model_ae = MultiModalAE(text_dim, latent_dim=LATENT_DIM).to(device)
optimiser = torch.optim.Adam(model_ae.parameters(), lr=1e-3)

loss_history_ae = []

for epoch in range(EPOCHS):
    total_loss = 0

    for (audio, text) in loader:
        audio = audio.to(device)
        text = text.to(device)

        optimiser.zero_grad()

        audio_recon, text_recon, mu = model_ae(audio, text)

        loss = multimodal_ae_loss(audio_recon, audio, text_recon, text)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    loss_history_ae.append(avg_loss)

    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')

# Save model and loss history
torch.save(model_ae.state_dict(), 'results_hard/conv_vae.pth')
np.save('results_hard/loss_history_ae.npy', loss_history_ae)

In [ ]:
# Plot training loss and save figure
plot_training_loss(loss_history_ae, 'results_hard/plots/loss_curve_ae.svg')

In [ ]:
# Get VAE model outputs for evaluation
model_bvae2.eval()
with torch.no_grad():
    mu_a_bvae2, _ = model_bvae2.encode_audio(audio_tensor.to(device))
    mu_t_bvae2, _ = model_bvae2.encode_text(lyrics_tensor.to(device))

    # Simply take mean of audio and text
    Z_bvae2 = ((mu_a_bvae2 + mu_t_bvae2) / 2).cpu().numpy()

model_bvae4.eval()
with torch.no_grad():
    mu_a_bvae4, _ = model_bvae4.encode_audio(audio_tensor.to(device))
    mu_t_bvae4, _ = model_bvae4.encode_text(lyrics_tensor.to(device))

    # Simply take mean of audio and text
    Z_bvae4 = ((mu_a_bvae4 + mu_t_bvae4) / 2).cpu().numpy()

# Get AE model outputs for evaluation
model_ae.eval()
with torch.no_grad():
    mu_a_ae = model_ae.encode_audio(audio_tensor.to(device))
    mu_t_ae = model_ae.encode_text(lyrics_tensor.to(device))

    # Simply take mean of audio and text
    Z_ae = ((mu_a_ae + mu_t_ae) / 2).cpu().numpy()

In [ ]:
# Flatten audio spectrograms for PCA
audio_flat = audio_X.reshape(audio_X.shape[0], -1)

# Normalize audio before PCA
audio_flat_scaled = StandardScaler().fit_transform(audio_flat)

# Raw combined for baseline comparison
Z_raw_combined = np.concatenate([audio_flat_scaled, lyrics_X], axis=1)

# PCA
pca = PCA(n_components=LATENT_DIM, random_state=27)
Z_pca_combined = pca.fit_transform(Z_raw_combined)

In [ ]:
results = []

for k, label_type, labels in [
    (2, 'language', languages),
    (6, 'genre', genres)
]:
    # Text only features
    (
        _, _, _, _,
        sil_kmeans_text, ch_kmeans_text, d_kmeans_text, ari_kmeans_text, nmi_kmeans_text, cluster_kmeans_text,
        sil_agglo_text, ch_agglo_text, d_agglo_text, ari_agglo_text, nmi_agglo_text, cluster_agglo_text,
    ) = evaluation_hard(lyrics_X, labels, k)
    
    # Raw features
    (
        _, _, _, _,
        sil_kmeans_raw, ch_kmeans_raw, d_kmeans_raw, ari_kmeans_raw, nmi_kmeans_raw, cluster_kmeans_raw,
        sil_agglo_raw, ch_agglo_raw, d_agglo_raw, ari_agglo_raw, nmi_agglo_raw, cluster_agglo_raw,
    ) = evaluation_hard(Z_raw_combined, labels, k)
    
    # PCA features
    (
        _, _, _, _,
        sil_kmeans_pca, ch_kmeans_pca, d_kmeans_pca, ari_kmeans_pca, nmi_kmeans_pca, cluster_kmeans_pca,
        sil_agglo_pca, ch_agglo_pca, d_agglo_pca, ari_agglo_pca, nmi_agglo_pca, cluster_agglo_pca,
    ) = evaluation_hard(Z_pca_combined, labels, k)

    # AE features
    (
        _, _, _, _,
        sil_kmeans_ae, ch_kmeans_ae, d_kmeans_ae, ari_kmeans_ae, nmi_kmeans_ae, cluster_kmeans_ae,
        sil_agglo_ae, ch_agglo_ae, d_agglo_ae, ari_agglo_ae, nmi_agglo_ae, cluster_agglo_ae,
    ) = evaluation_hard(Z_ae, labels, k)
    
    # Beta-VAE (Beta=2) features
    (
        _, _, _, _,
        sil_kmeans_bvae2, ch_kmeans_bvae2, d_kmeans_bvae2, ari_kmeans_bvae2, nmi_kmeans_bvae2, cluster_kmeans_bvae2,
        sil_agglo_bvae2, ch_agglo_bvae2, d_agglo_bvae2, ari_agglo_bvae2, nmi_agglo_bvae2, cluster_agglo_bvae2,
    ) = evaluation_hard(Z_bvae2, labels, k)

    # Beta-VAE (Beta=4) features
    (
        _, _, _, _,
        sil_kmeans_bvae4, ch_kmeans_bvae4, d_kmeans_bvae4, ari_kmeans_bvae4, nmi_kmeans_bvae4, cluster_kmeans_bvae4,
        sil_agglo_bvae4, ch_agglo_bvae4, d_agglo_bvae4, ari_agglo_bvae4, nmi_agglo_bvae4, cluster_agglo_bvae4,
    ) = evaluation_hard(Z_bvae4, labels, k)

    # Add to results
    results.append([
        k, label_type, 'TEXT',
        sil_kmeans_text, ch_kmeans_text, d_kmeans_text, ari_kmeans_text, nmi_kmeans_text, cluster_kmeans_text,
        sil_agglo_text, ch_agglo_text, d_agglo_text, ari_agglo_text, nmi_agglo_text, cluster_agglo_text,
    ])
    results.append([
        k, label_type, 'RAW',
        sil_kmeans_raw, ch_kmeans_raw, d_kmeans_raw, ari_kmeans_raw, nmi_kmeans_raw, cluster_kmeans_raw,
        sil_agglo_raw, ch_agglo_raw, d_agglo_raw, ari_agglo_raw, nmi_agglo_raw, cluster_agglo_raw,
    ])
    results.append([
        k, label_type, 'PCA',
        sil_kmeans_pca, ch_kmeans_pca, d_kmeans_pca, ari_kmeans_pca, nmi_kmeans_pca, cluster_kmeans_pca,
        sil_agglo_pca, ch_agglo_pca, d_agglo_pca, ari_agglo_pca, nmi_agglo_pca, cluster_agglo_pca,
    ])
    results.append([
        k, label_type, 'AE',
        sil_kmeans_ae, ch_kmeans_ae, d_kmeans_ae, ari_kmeans_ae, nmi_kmeans_ae, cluster_kmeans_ae,
        sil_agglo_ae, ch_agglo_ae, d_agglo_ae, ari_agglo_ae, nmi_agglo_ae, cluster_agglo_ae,
    ])
    results.append([
        k, label_type, 'Beta-VAE (Beta=2)',
        sil_kmeans_bvae2, ch_kmeans_bvae2, d_kmeans_bvae2, ari_kmeans_bvae2, nmi_kmeans_bvae2, cluster_kmeans_bvae2,
        sil_agglo_bvae2, ch_agglo_bvae2, d_agglo_bvae2, ari_agglo_bvae2, nmi_agglo_bvae2, cluster_agglo_bvae2,
    ])
    results.append([
        k, label_type, 'Beta-VAE (Beta=4)',
        sil_kmeans_bvae4, ch_kmeans_bvae4, d_kmeans_bvae4, ari_kmeans_bvae4, nmi_kmeans_bvae4, cluster_kmeans_bvae4,
        sil_agglo_bvae4, ch_agglo_bvae4, d_agglo_bvae4, ari_agglo_bvae4, nmi_agglo_bvae4, cluster_agglo_bvae4,
    ])

df_results = pd.DataFrame(results, columns=[
    'k', 'label_type', 'method',
    'silhouette_kmeans', 'calinski_harabasz_kmeans', 'davies_bouldin_kmeans', 'ari_kmeans', 'nmi_kmeans', 'cluster_kmeans',
    'silhouette_agglo', 'calinski_harabasz_agglo', 'davies_bouldin_agglo', 'ari_agglo', 'nmi_agglo', 'cluster_agglo',
    ])
df_results.to_csv('results_hard/clustering_metrics.csv', index=False)

df_results

In [ ]:
plot_tsne(lyrics_X, languages, 't-SNE (Text Only) - Language', 'results_hard/plots/tsne_language_text.svg', cmap_name='coolwarm')
plot_tsne(Z_raw_combined, languages, 't-SNE (Raw) - Language', 'results_hard/plots/tsne_language_raw.svg', cmap_name='coolwarm')
plot_tsne(Z_pca_combined, languages, 't-SNE (PCA) - Language', 'results_hard/plots/tsne_language_pca.svg', cmap_name='coolwarm')
plot_tsne(Z_ae, languages, 't-SNE (AE) - Language', 'results_hard/plots/tsne_language_ae.svg', cmap_name='coolwarm')
plot_tsne(Z_bvae2, languages, 't-SNE (Beta-VAE, Beta=2) - Language', 'results_hard/plots/tsne_language_bvae2.svg', cmap_name='coolwarm')
plot_tsne(Z_bvae4, languages, 't-SNE (Beta-VAE, Beta=4) - Language', 'results_hard/plots/tsne_language_bvae4.svg', cmap_name='coolwarm')

plot_tsne(lyrics_X, genres, 't-SNE (Text Only) - Genre', 'results_hard/plots/tsne_genre_text.svg', cmap_name='tab10')
plot_tsne(Z_raw_combined, genres, 't-SNE (Raw) - Genre', 'results_hard/plots/tsne_genre_raw.svg', cmap_name='tab10')
plot_tsne(Z_pca_combined, genres, 't-SNE (PCA) - Genre', 'results_hard/plots/tsne_genre_pca.svg', cmap_name='tab10')
plot_tsne(Z_ae, genres, 't-SNE (AE) - Genre', 'results_hard/plots/tsne_genre_ae.svg', cmap_name='tab10')
plot_tsne(Z_bvae2, genres, 't-SNE (Beta-VAE, Beta=2) - Genre', 'results_hard/plots/tsne_genre_bvae2.svg', cmap_name='tab10')
plot_tsne(Z_bvae4, genres, 't-SNE (Beta-VAE, Beta=4) - Genre', 'results_hard/plots/tsne_genre_bvae4.svg', cmap_name='tab10')